In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import platform

import torch
import torchvision

print(platform.python_version(), torch.__version__, torchvision.__version__, torch.version.cuda)
assert torch.cuda.is_available()

In [ ]:
from google.colab import drive, userdata

token = userdata.get("GH_TOKEN")
!git clone -q https://{token}@github.com/duyenphung206-sudo/vit-oxford-pets.git /content/vit-oxford-pets
%cd /content/vit-oxford-pets
!pip install -q -r requirements.txt
drive.mount("/content/drive")

In [ ]:
from torchvision import models

weights = models.ViT_B_16_Weights.IMAGENET1K_V1
vit = models.vit_b_16(weights=weights)
n_params = sum(p.numel() for p in vit.parameters())
print(f"{n_params:,}", weights.meta["_metrics"]["ImageNet-1K"])
print(weights.transforms())
assert n_params == 86_567_656

In [ ]:
from src.models.factory import build_model
from src.utils import count_parameters

for strategy, k in [("linear_probe", 0), ("partial", 4), ("full", 0)]:
    cfg = {"name": "vit_b_16", "num_classes": 37, "strategy": strategy, "unfreeze_last_k": k}
    p = count_parameters(build_model(cfg))
    print(f"{strategy:<13} {p['trainable']:>11,} / {p['total']:,}")

In [ ]:
import time

device = torch.device("cuda")
model = build_model({"name": "vit_b_16", "num_classes": 37, "strategy": "full"}).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.003, momentum=0.9)
scaler = torch.amp.GradScaler("cuda")
loss_fn = torch.nn.CrossEntropyLoss()

for batch_size in (32, 64):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    images = torch.randn(batch_size, 3, 224, 224, device=device)
    labels = torch.randint(0, 37, (batch_size,), device=device)
    timings = []
    for _ in range(6):
        torch.cuda.synchronize()
        start = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda"):
            loss = loss_fn(model(images), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        torch.cuda.synchronize()
        timings.append(time.perf_counter() - start)
    step_ms = 1000 * sum(timings[1:]) / len(timings[1:])
    print(f"batch {batch_size:>3} | {step_ms:6.0f} ms/step | {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")

In [ ]:
!python -m src.train --config configs/e4_vit_full_ft.yaml --fake-data --epochs 1 --output-dir /content/smoke --checkpoint-dir /content/smoke_ckpt
!cat /content/smoke/metrics.json

In [ ]:
!pip freeze | grep -i -E "^(torch|torchvision|numpy|pandas|matplotlib|scikit-learn|pyyaml|tqdm|pillow|gradio|pytest)=="